# LifeMate: Unsloth QLoRA global → local, 2 GPUs

Chọn **GPU T4 ×2**, bật Internet, rồi Run All. Notebook này được lưu để bạn tự chạy; việc lưu không khởi chạy GPU.

- Qwen3-4B-Instruct-2507, NF4 QLoRA r=16, chỉ fine-tune LLM; ASR được giữ để inference.
- **DDP 2 GPU; micro-batch train = 4/GPU; validation loss = 4/GPU; accumulation = 1; effective train batch = 8.**
- Global tối đa **1 epoch**; planner/caller 1 epoch; summarizer 2 epochs. Ba local adapter độc lập khởi tạo từ global tốt nhất.
- Cosine LR, warmup 3%; eval mỗi 50 optimizer attempts và cuối stage; early stopping patience 3, min_delta .001; khôi phục checkpoint loss tốt nhất.
- Train 1.445 yêu cầu; valid đúng 100 ID từ split.txt. Rationale Codex đã cung cấp, không dùng Qwen làm teacher.
- Sau train: ASR cho audio hiện tại và history → vLLM multi-LoRA → ToolEnv → chấm valid; **batch inference = 4**, một GPU, không giới hạn thời gian mỗi mẫu.
- Prompt không chứa gold valid, oracle transcript, few-shot valid hoặc luật theo ID. Valid gold chỉ dùng tính loss và chấm điểm.
- Test CPU/Gloo 2 rank kiểm tra token-weighted gradients, uneven validation, checkpoint restoration và early stopping. **Bản DDP Qwen này chưa được chạy thử trên GPU**. Notebook tự kiểm tra imports/NCCL/AMP trước khi load Qwen.

Checkpoint chỉ lưu adapter tốt nhất để đánh giá/tiếp tục từng stage; không hỗ trợ resume optimizer giữa một stage. Chọn RUN_NAME mới khi chạy lại một stage đã có checkpoint.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,zipfile
REPO_URL = 'https://github.com/tnisl/baseline.git'
# Frozen core; all changed runtime sources and package locks are embedded below.
GIT_REF = '4894d331692293565635454a8c686b7c18314a04'
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
MODEL_REVISION = 'cdbee75f17c01a7cc42f958dc650907174af0554'
WORLD_SIZE = 2
TRAIN_MICRO_BATCH = 4
VALID_LOSS_BATCH = 4
INFERENCE_BATCH = 4
GRADIENT_ACCUMULATION = 1
EPOCHS = {'global':1,'planner':1,'caller':1,'summarizer':2}
RUN_NAME = 'full_2gpu_batch4_v1'
MODES = ('multi',)  # Optional: ('single','multi') for the global-vs-local comparison.
VALID_LIMIT = None  # None=all 100; optionally 4..100 for shorter post-training inference only.
ASR_THREADS = 4
assert WORLD_SIZE==2 and TRAIN_MICRO_BATCH==VALID_LOSS_BATCH==INFERENCE_BATCH==4
assert GRADIENT_ACCUMULATION==1 and EPOCHS['global']==1
assert set(MODES)<={'single','multi'} and MODES
assert VALID_LIMIT is None or 4<=VALID_LIMIT<=100
hardware=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv,noheader'],text=True)
assert len(hardware.strip().splitlines())==2,'Select GPU T4 x2 in Kaggle notebook settings.'
print(hardware)
SOURCE=Path('/kaggle/working/paper_finetune_2gpu_source')
WORK=Path('/kaggle/working/paper_finetune_2gpu')/RUN_NAME
DATA=WORK/'prepared';DATA.mkdir(parents=True,exist_ok=True)
if not SOURCE.exists():
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin',GIT_REF],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
assert subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()==GIT_REF


In [ ]:
PAYLOAD={'umilife/ddp_engine.py': '"""Explicit, symmetric DDP collectives; no Trainer monkey-patched control flow."""\nimport contextlib,hashlib,json,math,random,shutil,time\nfrom pathlib import Path\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\n\n\ndef batches(lengths,epoch,rank,world,batch_size=4):\n    # Shuffle then length-bucket across both GPUs to reduce padding/stragglers.\n    indices=list(range(len(lengths)));random.Random(42+epoch).shuffle(indices)\n    width=batch_size*world\n    ordered=[]\n    for start in range(0,len(indices),width*32):\n        ordered.extend(sorted(indices[start:start+width*32],key=lambda i:lengths[i],reverse=True))\n    padding=(-len(ordered))%width\n    ordered.extend(ordered[:padding])\n    return [ordered[i+rank*batch_size:i+(rank+1)*batch_size] for i in range(0,len(ordered),width)]\n\n\ndef state_digest(model):\n    h=hashlib.sha256()\n    for name,p in sorted(model.named_parameters()):\n        if p.requires_grad:\n            h.update(name.encode());h.update(p.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())\n    return h.hexdigest()\n\n\ndef agreement(value):\n    values=[None]*dist.get_world_size();dist.all_gather_object(values,value)\n    assert len(set(values))==1,values\n    return value\n\n\ndef weighted_eval(model,items,collate,device,amp):\n    # No DDP forward: rank workloads can differ by one validation example.\n    model.eval();rank=dist.get_rank();world=dist.get_world_size()\n    totals=torch.zeros(2,device=device,dtype=torch.float64)\n    local=list(range(rank,len(items),world))\n    with torch.inference_mode():\n        for start in range(0,len(local),4):\n            x={k:v.to(device) for k,v in collate([items[i] for i in local[start:start+4]]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            with amp():loss=model(**x,use_cache=False,return_dict=True).loss\n            totals[0]+=loss.double()*n;totals[1]+=n\n    dist.all_reduce(totals)\n    result=(totals[0]/totals[1]).item()\n    assert math.isfinite(result) and totals[1]>0,\'Nonfinite validation loss\'\n    model.train()\n    return result\n\n\nclass LossModule(torch.nn.Module):\n    # DDP static_graph must see the loss tensor directly even for custom ModelOutput objects.\n    def __init__(self,model):super().__init__();self.model=model\n    def forward(self,**kwargs):return self.model(**kwargs).loss\n\n\ndef fit(model,train,valid,collate,device,output,epochs,lr,save_model,load_model,\n        eval_steps=50,patience=3,min_delta=.001,amp_dtype=None):\n    rank=dist.get_rank();world=dist.get_world_size();assert world==2\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    assert not (output/\'best\').exists(),\'Use a fresh run directory; completed checkpoints are not overwritten\'\n    params=[p for p in model.parameters() if p.requires_grad]\n    wrapped=DistributedDataParallel(LossModule(model),device_ids=[device.index] if device.type==\'cuda\' else None,\n        broadcast_buffers=False,find_unused_parameters=False,static_graph=True)\n    amp=(lambda:torch.autocast(device_type=device.type,dtype=amp_dtype)) if amp_dtype else contextlib.nullcontext\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=(device.type==\'cuda\' and amp_dtype==torch.float16),init_scale=1024)\n    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=.01)\n    lengths=[len(x[\'input_ids\']) for x in train]\n    steps_per_epoch=len(batches(lengths,0,rank,world));total=steps_per_epoch*epochs\n    warmup=max(1,math.ceil(.03*total))\n    def factor(step):\n        if step<warmup:return (step+1)/warmup\n        return .5*(1+math.cos(math.pi*min(1.,(step-warmup)/max(1,total-warmup))))\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,factor)\n    initial=agreement(state_digest(model))\n    before=weighted_eval(model,valid,collate,device,amp)\n    best=math.inf;significant=math.inf;bad=0;best_step=0;history=[]\n    step=0;updates=0;skipped=0;stop=False;started=time.monotonic()\n    last_epoch=0.;saved_hash=None\n    for epoch in range(epochs):\n        local_batches=batches(lengths,epoch,rank,world)\n        for batch_index,indices in enumerate(local_batches):\n            model.train();optimizer.zero_grad(set_to_none=True)\n            x={k:v.to(device) for k,v in collate([train[i] for i in indices]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            global_n=n.clone();dist.all_reduce(global_n)\n            with amp():\n                raw_loss=wrapped(**x,use_cache=False,return_dict=True)\n                # DDP averages rank gradients; recover true global token-mean gradient.\n                loss=raw_loss*(world*n/global_n)\n            finite=torch.isfinite(loss).to(torch.int32);dist.all_reduce(finite,op=dist.ReduceOp.MIN)\n            assert finite.item()==1,\'Nonfinite training loss on a rank\'\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(params,1.)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            overflow=scaler.get_scale()<old_scale\n            agreement(str(overflow))\n            if overflow:skipped+=1\n            else:scheduler.step();updates+=1\n            step+=1;last_epoch=epoch+(batch_index+1)/len(local_batches)\n            if step%10==0 or step==1:\n                report={\'step\':step,\'planned_steps\':total,\'epoch\':last_epoch,\'local_loss\':raw_loss.item(),\n                        \'lr\':optimizer.param_groups[0][\'lr\'],\'grad_norm\':float(norm),\'optimizer_updates\':updates}\n                if rank==0:print(json.dumps(report),flush=True)\n            at_end=step==total\n            if step%eval_steps==0 or at_end:\n                metric=weighted_eval(model,valid,collate,device,amp)\n                improved=metric<best\n                if metric<significant-min_delta:significant=metric;bad=0\n                else:bad+=1\n                # Every rank computes the same scalar/state; only I/O is rank-zero-only.\n                if improved:\n                    best=metric;best_step=step\n                    dist.barrier()\n                    if rank==0:\n                        temp=output/\'best.tmp\'\n                        if temp.exists():shutil.rmtree(temp)\n                        temp.mkdir();save_model(model,temp)\n                        (temp/\'checkpoint.json\').write_text(json.dumps({\'step\':step,\'eval_loss\':metric,\'epoch\':last_epoch}))\n                        if (output/\'best\').exists():shutil.rmtree(output/\'best\')\n                        temp.rename(output/\'best\')\n                    dist.barrier()\n                    saved_hash=agreement(state_digest(model))\n                event={\'step\':step,\'epoch\':last_epoch,\'eval_loss\':metric,\'best_step\':best_step,\n                       \'best_loss\':best,\'patience_used\':bad,\'improved\':improved}\n                history.append(event)\n                if rank==0:print(json.dumps(event),flush=True)\n                stop=bad>=patience\n                agreement(json.dumps([step,best,best_step,bad,stop]))\n                if stop:break\n        if stop:break\n    assert updates>0 and best_step>0\n    dist.barrier();load_model(model,output/\'best\');dist.barrier()\n    final=agreement(state_digest(model));assert final==saved_hash and final!=initial\n    after=weighted_eval(model,valid,collate,device,amp)\n    assert abs(after-best)<max(.005,abs(best)*.005),\'Best checkpoint restoration changed loss\'\n    result={\'world_size\':world,\'micro_batch\':4,\'eval_batch_per_device\':4,\'gradient_accumulation\':1,\n            \'effective_batch\':8,\'epochs\':epochs,\'actual_epoch\':last_epoch,\'global_step\':step,\n            \'optimizer_updates\':updates,\'amp_skipped_updates\':skipped,\'planned_steps\':total,\n            \'learning_rate\':lr,\'scheduler\':\'cosine\',\'warmup_ratio\':.03,\n            \'early_stopping\':{\'patience\':patience,\'min_delta\':min_delta,\'eval_steps\':eval_steps,\n                \'restore_best\':True,\'best_step\':best_step,\'best_metric\':best,\'stopped_early\':stop},\n            \'before\':{\'eval_loss\':before},\'after\':{\'eval_loss\':after},\'history\':history,\n            \'initial_trainable_sha256\':initial,\'final_trainable_sha256\':final,\n            \'wall_s\':time.monotonic()-started,\'train_padding_examples_per_epoch\':(-len(train))%8,\n            \'validation_repeats\':0}\n    dist.barrier();del wrapped\n    return result\n', 'umilife/train_2gpu.py': '"""Notebook runtime: two-rank Unsloth QLoRA, explicit DDP loop and best adapter."""\nimport os\nos.environ.setdefault(\'UNSLOTH_COMPILE_DISABLE\',\'1\')\nos.environ.setdefault(\'UNSLOTH_RETURN_LOGITS\',\'0\')\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\')\n\ndef main():\n    import argparse,json,hashlib,importlib.metadata\n    from pathlib import Path\n    import torch\n    local=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(local)\n    from unsloth import FastLanguageModel\n    import torch.distributed as dist\n    from datetime import timedelta\n    from peft import set_peft_model_state_dict\n    from peft.utils.save_and_load import load_peft_weights\n    from umilife.core import digest,write_json,dumps\n    from umilife.models import TARGET_MODULES,seed_all\n    from umilife.train import Dataset,Collator,adapter_state\n    from umilife.prompt_policy import PROMPTS,POLICY_AUDIT\n    from umilife.ddp_engine import fit,agreement\n    p=argparse.ArgumentParser()\n    for k in (\'stage\',\'data-dir\',\'output-dir\',\'model\',\'revision\'):p.add_argument(\'--\'+k,required=True)\n    p.add_argument(\'--global-adapter\');p.add_argument(\'--epochs\',type=int,required=True)\n    a=p.parse_args()\n    assert int(os.environ[\'WORLD_SIZE\'])==2 and torch.cuda.device_count()==2\n    assert a.stage in (\'global\',\'planner\',\'caller\',\'summarizer\')\n    assert a.epochs==1 if a.stage==\'global\' else a.epochs>0\n    assert bool(a.global_adapter)==(a.stage!=\'global\')\n    dist.init_process_group(\'nccl\',timeout=timedelta(minutes=20),device_id=torch.device(\'cuda\',local))\n    seed_all(42)\n    data=Path(a.data_dir);out=Path(a.output_dir);target=out/\'adapters\'/a.stage\n    assert not target.exists(),\'Refuse to overwrite completed adapter\'\n    manifest=json.loads((data/\'manifest.json\').read_text())\n    prompt_sha=POLICY_AUDIT[\'static_prompts_sha256\']\n    assert manifest[\'prompts_sha256\']==prompt_sha and manifest[\'rationale_authors\']==[\'Codex\']\n    for part in (\'train\',\'validation\'):\n        f=data/f\'{a.stage}.{part}.jsonl\';assert digest(f)==manifest[\'files_sha256\'][f.name]\n    dtype=torch.bfloat16 if torch.cuda.get_device_capability(local)[0]>=8 else torch.float16\n    model,tokenizer=FastLanguageModel.from_pretrained(model_name=a.model,revision=a.revision,\n        use_exact_model_name=True,max_seq_length=8192,dtype=dtype,load_in_4bit=True,\n        fast_inference=False,device_map={\'\':local},trust_remote_code=False)\n    model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,\n        target_modules=TARGET_MODULES,bias=\'none\',use_gradient_checkpointing=True,random_state=42)\n    parent=None\n    def load_adapter(model,path):\n        result=set_peft_model_state_dict(model,load_peft_weights(str(path),device=\'cpu\'))\n        assert not result.unexpected_keys,result.unexpected_keys\n    if a.global_adapter:\n        parent=Path(a.global_adapter);lineage=json.loads((parent/\'lineage.json\').read_text())\n        assert lineage[\'stage\']==\'global\' and lineage[\'model\']==a.model and lineage[\'revision\']==a.revision\n        assert lineage[\'dataset_manifest_sha256\']==digest(data/\'manifest.json\')\n        load_adapter(model,parent)\n    initial=adapter_state(model)\n    if parent:assert initial[\'sha256\']==lineage[\'final_adapter\'][\'sha256\'],\'Global initialization mismatch\'\n    assert all(\'lora_\' in n for n,v in model.named_parameters() if v.requires_grad)\n    FastLanguageModel.for_training(model)\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.config.use_cache=False\n    train=Dataset(data/f\'{a.stage}.train.jsonl\',tokenizer,8192)\n    valid=Dataset(data/f\'{a.stage}.validation.jsonl\',tokenizer,8192)\n    assert train.stats[\'overlength\']==valid.stats[\'overlength\']==0,\'No training examples may be silently dropped\'\n    def save_adapter(model,path):model.save_pretrained(path,safe_serialization=True)\n    result=fit(model,train,valid,Collator(tokenizer.pad_token_id),torch.device(\'cuda\',local),\n        out/\'checkpoints\'/a.stage,a.epochs,5e-5 if a.stage==\'global\' else 1e-5,\n        save_adapter,load_adapter,amp_dtype=dtype)\n    final=adapter_state(model);assert final[\'finite\'] and final[\'sha256\']!=initial[\'sha256\']\n    agreement(final[\'sha256\'])\n    if dist.get_rank()==0:\n        save_adapter(model,target);tokenizer.save_pretrained(target)\n        history=result.pop(\'history\');write_json(target/\'training_history.json\',history)\n        record={**result,\'stage\':a.stage,\'model\':a.model,\'revision\':a.revision,\n            \'method\':\'Unsloth QLoRA GLPFT\',\'initial_adapter\':initial,\'final_adapter\':final,\n            \'parent\':str(parent) if parent else None,\n            \'global_adapter_sha256\':digest(parent/\'adapter_model.safetensors\') if parent else None,\n            \'dataset_manifest_sha256\':digest(data/\'manifest.json\'),\'prompt_sha256\':prompt_sha,\n            \'tokenization\':{\'train\':train.stats,\'validation\':valid.stats},\n            \'peak_cuda_gb_rank0\':torch.cuda.max_memory_allocated()/2**30,\n            \'versions\':{k:importlib.metadata.version(k) for k in (\'torch\',\'transformers\',\'unsloth\',\'unsloth_zoo\',\'peft\',\'bitsandbytes\')}}\n        write_json(target/\'lineage.json\',record);print(dumps(record),flush=True)\n    dist.barrier();dist.destroy_process_group()\nif __name__==\'__main__\':main()\n', 'umilife/prepare_2gpu.py': '"""Fixed split, existing Codex rationales, prior-evidence-only training prompts."""\nimport argparse,hashlib,json\nfrom copy import deepcopy\nfrom collections import Counter\nfrom pathlib import Path\nfrom jsonschema import Draft202012Validator\nfrom umilife.core import read_jsonl,write_json,dumps,digest,allowed_specs,load_catalog,import_toolenv\nfrom umilife.data import annotation_tasks,normalized_row\nfrom umilife.prompt_policy import PROMPTS,build_messages,request_schema,POLICY_AUDIT\nfrom umilife.train import encode\n\nRATIONAL_SHA=\'d14768df60e155521a57998a5c6d0c4c1cc3b6372b74cce7a955098fd3fdf497\'\nSPLIT_SHA=\'631f8cfab61ddde833477f743cf1a65a9012526e0178e97046a36ee864e3d505\'\nTRAIN_SHA=\'24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8\'\n\ndef brief_facts(value,prefix=\'\',budget=8):\n    if isinstance(value,dict):\n        out=[]\n        for key,item in value.items():\n            out.extend(brief_facts(item,f\'{prefix}.{key}\'.strip(\'.\'),budget-len(out)))\n            if len(out)>=budget:break\n        return out\n    if isinstance(value,list):\n        out=[f\'{prefix}: {len(value)} mục\']\n        for i,item in enumerate(value[:2]):\n            out.extend(brief_facts(item,f\'{prefix}[{i}]\',budget-len(out)))\n            if len(out)>=budget:break\n        return out[:budget]\n    if budget<=0:return []\n    text=str(value) if value is not None else \'null\'\n    return [prefix+\'=\'+text[:120]]\n\ndef terminal(trajectory,tokenizer):\n    if not trajectory:\n        # Existing dataset convention: zero calls -> give_up. Includes some no-op/cancellation cases.\n        return {\'thought\':\'Yêu cầu không cần hoặc không thể xử lý bằng các công cụ được cài.\',\'next\':\'give_up\'},None\n    last=trajectory[-1];action=last[\'action\']\n    if action[\'name\']==\'sys.ask_missing_info\':\n        answer=\'Bạn vui lòng bổ sung \'+\', \'.join(action[\'arguments\'].get(\'params\',[]))+\' cho \'+action[\'arguments\'].get(\'tool_name\',\'yêu cầu\')+\'.\'\n        thought=\'Đã hỏi thông tin còn thiếu, cần chờ người dùng trả lời.\'\n    else:\n        # Deterministic grounded label, not an LLM teacher and not a full observation dump.\n        answer=\'Kết quả \'+action[\'name\']+\': \'+\'; \'.join(brief_facts(last[\'observation\']))+\'.\'\n        thought=\'Đã có kết quả của các bước trong nhãn, chuyển sang tóm tắt bằng chứng.\'\n    answer=tokenizer.decode(tokenizer.encode(answer[:800],add_special_tokens=False)[:180],skip_special_tokens=True)\n    return {\'thought\':thought,\'next\':\'summarizer\'},answer\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'output\',\'rationales\',\'split-file\',\'model\',\'revision\'):p.add_argument(\'--\'+name,required=True)\n    a=p.parse_args();root=Path(a.data_root);out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    assert digest(root/\'train/train.jsonl\')==TRAIN_SHA\n    assert digest(a.rationales)==RATIONAL_SHA and digest(a.split_file)==SPLIT_SHA\n    from transformers import AutoTokenizer\n    tokenizer=AutoTokenizer.from_pretrained(a.model,revision=a.revision)\n    ids=Path(a.split_file).read_text().splitlines();valid=set(ids);assert len(ids)==len(valid)==100\n    rows=read_jsonl(root/\'train/train.jsonl\');assert len(rows)==1545\n    assignments={r[\'id\']:\'validation\' if r[\'id\'] in valid else \'train\' for r in rows}\n    assert valid=={k for k,v in assignments.items() if v==\'validation\'}\n    records=read_jsonl(a.rationales);assert len(records)==3541 and {r[\'author\'] for r in records}=={\'Codex\'}\n    rationales={r[\'key\']:r[\'thought\'] for r in records};assert len(rationales)==len(records)\n    catalog=load_catalog(root,\'train\');ToolEnv=import_toolenv(root,\'train\')\n    tasks={(t[\'id\'],t[\'step\']):t for t in annotation_tasks(rows,catalog,assignments)}\n    counts=Counter();references=[];token_stats={};schema_cache={}\n    handles={(role,part):(out/f\'{role}.{part}.jsonl\').open(\'w\') for role in (\'global\',\'planner\',\'caller\',\'summarizer\') for part in (\'train\',\'validation\')}\n    def add(role,row,specs,trajectory,target,plan=None):\n        part=assignments[row[\'id\']]\n        schema=request_schema(role,specs,plan);key=dumps(schema)\n        if key not in schema_cache:schema_cache[key]=Draft202012Validator(schema)\n        schema_cache[key].validate(target)\n        rec={\'id\':row[\'id\'],\'role\':role,\'step\':len(trajectory),\n             \'messages\':build_messages(role,normalized_row(row),row[\'audio_text\'],specs,trajectory,plan),\n             \'target\':dumps(target)}\n        encoded=encode(tokenizer,rec);n=len(encoded[\'input_ids\']);assert n<=8192,(\'Overlength\',row[\'id\'],role,n)\n        stat=token_stats.setdefault(f\'{role}.{part}\',{\'records\':0,\'max_tokens\':0,\'max_target_tokens\':0})\n        stat[\'records\']+=1;stat[\'max_tokens\']=max(stat[\'max_tokens\'],n)\n        stat[\'max_target_tokens\']=max(stat[\'max_target_tokens\'],sum(x!=-100 for x in encoded[\'labels\']))\n        handles[role,part].write(dumps(rec)+\'\\n\');counts[f\'{role}.{part}\']+=1\n    try:\n        for row in rows:\n            specs=allowed_specs(row,catalog);env=ToolEnv(\'train\',row[\'id\'],root=root);trajectory=[]\n            gold=row[\'label\'][\'tool_calls\'];results=row[\'label\'][\'tool_results\'];assert len(gold)==len(results)\n            for step,(action,expected) in enumerate(zip(gold,results,strict=True)):\n                assert env.validate(action[\'name\'],action[\'arguments\']).get(\'ok\')\n                observed=env.call(action[\'name\'],action[\'arguments\']);assert observed==expected,(row[\'id\'],step)\n                thought=rationales[tasks[row[\'id\'],step][\'key\']]\n                plan={\'thought\':thought,\'next\':\'caller\',\'tool\':action[\'name\']}\n                add(\'planner\',row,specs,trajectory,plan)\n                add(\'caller\',row,specs,trajectory,action,plan)\n                add(\'global\',row,specs,trajectory,{**plan,\'action\':action})\n                trajectory.append({\'thought\':thought,\'action\':action,\'observation\':observed})\n            references.append({\'id\':row[\'id\'],\'tool_calls\':gold,\'state_hash\':env.get_state_hash()})\n            plan,answer=terminal(trajectory,tokenizer)\n            add(\'planner\',row,specs,trajectory,plan)\n            add(\'global\',row,specs,trajectory,{**plan,**({\'answer\':answer} if answer is not None else {})})\n            if answer is not None:add(\'summarizer\',row,specs,trajectory,{\'answer\':answer},plan)\n    finally:\n        for f in handles.values():f.close()\n    assert counts=={\'global.train\':4759,\'planner.train\':4759,\'caller.train\':3314,\'summarizer.train\':1334,\n                    \'global.validation\':327,\'planner.validation\':327,\'caller.validation\':227,\'summarizer.validation\':100},dict(counts)\n    manifest={\'split\':assignments,\'counts\':dict(counts),\'excluded\':[],\n        \'train_episodes\':1445,\'validation_episodes\':100,\'split_sha256\':SPLIT_SHA,\'train_sha256\':TRAIN_SHA,\n        \'model\':a.model,\'revision\':a.revision,\'rationale_authors\':[\'Codex\'],\'rationales_sha256\':RATIONAL_SHA,\n        \'rationale_provenance\':\'Existing Codex-authored tool explanation templates plus episode overrides; unchanged thought strings. Ignore stale partition field.\',\n        \'summary_labels\':\'Deterministic concise facts from final gold tool observation; max 180 answer tokens; not human-written summaries.\',\n        \'zero_action_convention\':\'Preserved give_up target for all zero-call gold episodes, including some no-op/cancellation cases.\',\n        \'validation_usage\':\'Held-out labels only for loss evaluation and post-generation scoring; never optimizer updates or inference prompts.\',\n        \'training_transcripts\':\'Provided clean audio_text; post-training evaluation uses supplied ASR with no oracle fallback.\',\n        \'prompts_sha256\':POLICY_AUDIT[\'static_prompts_sha256\'],\'policy\':POLICY_AUDIT,\n        \'tokenization\':token_stats,\'files_sha256\':{f.name:digest(f) for f in out.glob(\'*.jsonl\')}}\n    write_json(out/\'references.json\',references);write_json(out/\'manifest.json\',manifest)\n    print(dumps(manifest),flush=True)\nif __name__==\'__main__\':main()\n', 'umilife/preflight_2gpu.py': '"""Executed by the user before Qwen loading: imports, NCCL and AMP backward on both GPUs."""\nimport os,json\nimport torch\nlocal=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(local)\nfrom unsloth import FastLanguageModel\nimport torch.distributed as dist\nfrom datetime import timedelta\nfrom torch.nn.parallel import DistributedDataParallel\nfrom umilife.ddp_engine import agreement\n\ndef main():\n    assert torch.cuda.device_count()==2 and int(os.environ[\'WORLD_SIZE\'])==2\n    dist.init_process_group(\'nccl\',device_id=torch.device(\'cuda\',local),timeout=timedelta(minutes=5))\n    dtype=torch.bfloat16 if torch.cuda.get_device_capability(local)[0]>=8 else torch.float16\n    model=torch.nn.Linear(16,8).to(torch.device(\'cuda\',local));wrapped=DistributedDataParallel(model,device_ids=[local],static_graph=True)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=.001)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=dtype==torch.float16,init_scale=1024)\n    for _ in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(\'cuda\',dtype=dtype):loss=wrapped(torch.randn(4,16,device=torch.device(\'cuda\',local))).float().square().mean()\n        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n    assert all(torch.isfinite(p).all() for p in model.parameters())\n    x=torch.tensor([1.],device=torch.device(\'cuda\',local));dist.all_reduce(x);assert x.item()==2\n    agreement(\'NCCL_AMP_BATCH4_PASS\')\n    if dist.get_rank()==0:print(json.dumps({\'status\':\'pass\',\'world_size\':2,\'micro_batch\':4,\'backend\':\'nccl\',\'amp\':str(dtype)}),flush=True)\n    dist.barrier();dist.destroy_process_group()\nif __name__==\'__main__\':main()\n', 'umilife/infer_2gpu.py': '"""Fine-tuned multi-LoRA workflow; held-out labels are never model inputs."""\nimport os\nos.environ.setdefault("CUDA_VISIBLE_DEVICES","0")\nos.environ.setdefault("VLLM_USE_V1","0")\nos.environ.setdefault("VLLM_ATTENTION_BACKEND","XFORMERS")\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD","spawn")\nos.environ.setdefault("VLLM_USAGE_STATS_ENABLED","0")\nos.environ.setdefault("DO_NOT_TRACK","1")\nimport json,time,argparse,math\nfrom pathlib import Path\nfrom collections import Counter\nfrom umilife.core import read_jsonl,write_json,dumps,parse_json,allowed_specs,load_catalog,import_toolenv\nfrom umilife.prompt_policy import (PROMPTS,COMPLETION_REVIEW,POLICY_AUDIT,POLICY_VERSION,\n    build_messages,request_schema,validate_meta_action,repeated_read,canonical,digest)\nfrom copy import deepcopy\nSYSTEMS=PROMPTS\nfrom umilife.data import normalized_row\nfrom umilife.grammar import role_schema\n\ndef rollout(row,text,specs,env,mode,max_calls=10,max_decisions=16):\n    started=time.monotonic();trajectory=[];trace=[];answer=None;reason="decision_limit"\n    feedback=None;failures=0;reviews=0;read_skips=0;repairs=0\n    def generate(role,plan=None,feedback=None):\n        schema=request_schema(role,specs,plan)\n        response=yield {"role":role,"messages":build_messages(role,row,text,specs,trajectory,plan,feedback),"schema":schema}\n        trace.append({"role":role,**response})\n        if response.get("schema_valid") is False:\n            raise ValueError("Model output violates request schema: "+response.get("schema_error",""))\n        return parse_json(response["raw"])\n    for _ in range(max_decisions):\n        try:\n            role="global" if mode=="single" else "planner"\n            planner_feedback=feedback\n            plan=yield from generate(role,feedback=planner_feedback);feedback=None\n            trace.append({"role":role,"output":plan})\n            if plan["next"] in ("summarizer","give_up"):\n                reviews+=1\n                trace.append({"event":"completion_review","proposal":plan})\n                plan=yield from generate(role,feedback={"kind":"completion_review",\'proposal\':plan,\'instruction\':COMPLETION_REVIEW})\n                trace.append({"role":role,"output":plan,"reviewed":True})\n            route=plan["next"]\n            if route=="give_up":reason="give_up";break\n            if route=="summarizer":\n                answer=plan.get("answer") if mode=="single" else (yield from generate("summarizer",plan)).get("answer")\n                reason="complete";break\n            if route!="caller" or plan.get("tool") not in specs:raise ValueError("Invalid planner route or unavailable tool")\n            if len(env.get_tool_calls())>=max_calls:reason="call_limit";break\n            action_feedback=planner_feedback;action=None\n            for attempt in range(3):\n                try:\n                    candidate=plan.get("action") if mode=="single" and attempt==0 else (\n                        (yield from generate(role,feedback=action_feedback)).get("action") if mode=="single" else\n                        (yield from generate("caller",plan,action_feedback)))\n                    if not isinstance(candidate,dict):raise ValueError("Invalid action object")\n                    name,args=candidate.get("name"),candidate.get("arguments")\n                    if name!=plan[\'tool\']:raise ValueError("Caller must use the tool chosen by Planner")\n                    check=env.validate(name,args) if name in specs and isinstance(args,dict) else {"ok":False,"error":"Unknown tool or invalid arguments"}\n                    if check.get(\'ok\'):\n                        extra=validate_meta_action(candidate,specs)\n                        if extra:check={\'ok\':False,\'error\':extra}\n                    trace.append({"role":"caller","output":candidate,"validation":check})\n                    if check.get(\'ok\'):action=candidate;break\n                    error=check.get(\'error\',\'Invalid arguments\')\n                except (ValueError,KeyError,TypeError) as exc:error=str(exc)\n                repairs+=1\n                action_feedback={"kind":"tool_contract_error","error":error,\n                    "instruction":"Sửa theo schema và dữ kiện đã có; nếu thiếu căn cứ thì cần lập kế hoạch tra cứu hoặc hỏi bổ sung."}\n            if action is None:\n                failures+=1;feedback=action_feedback\n                if failures>=3:reason="format_error";trace.append({"error":"tool contract repair exhausted"});break\n                continue\n            if repeated_read(action,specs,trajectory):\n                read_skips+=1\n                feedback={\'kind\':\'repeated_read\',\'action\':action,\n                    \'instruction\':\'Kết quả lời gọi này đã có trong trajectory và chưa có thao tác ghi làm thay đổi dữ liệu. Dùng kết quả đó; nếu rỗng hãy đổi bộ lọc theo schema hoặc hỏi thiếu thông tin, không lặp nguyên lời gọi.\'}\n                trace.append({\'event\':\'repeat_read_prevented\',\'action\':action});continue\n            name,args=action[\'name\'],action[\'arguments\']\n            observed=env.call(name,args);failures=0\n            trajectory.append({\'step\':len(trajectory),\'thought\':plan.get(\'thought\',\'\'),\'action\':action,\'observation\':observed})\n            if name==\'sys.ask_missing_info\':\n                terminal={\'thought\':\'Đã yêu cầu bổ sung thông tin; chờ người dùng trả lời.\',\'next\':\'summarizer\'}\n                answer=(yield from generate(\'summarizer\',terminal)).get(\'answer\')\n                reason=\'needs_user_input\';break\n        except (ValueError,KeyError,TypeError) as exc:\n            failures+=1;repairs+=1;trace.append({\'error\':str(exc)})\n            if failures>=3:reason=\'format_error\';break\n            feedback={\'kind\':\'planning_contract_error\',\'error\':str(exc),\'instruction\':\'Lập lại đúng một bước theo dữ kiện và schema công cụ đã cài.\'}\n    return {\'id\':row[\'id\'],\'tool_calls\':env.get_tool_calls(),\'state_hash\':env.get_state_hash()},{\n        \'id\':row[\'id\'],\'reason\':reason,\'elapsed_s\':time.monotonic()-started,\n        \'over_deadline\':False,\'time_limit_enforced\':False,\'answer\':answer,\n        \'trace\':trace,\'trajectory\':trajectory,\'transcript\':text,\n        \'completion_reviews\':reviews,\'repeat_reads_prevented\':read_skips,\'repairs\':repairs,\'prompt_policy\':POLICY_VERSION}\n\ndef run_batch(generators,backend):\n    active={}\n    completed={}\n    for index,generator in enumerate(generators):\n        try:active[index]=next(generator)\n        except StopIteration as stop:completed[index]=stop.value\n    while active:\n        good=[];bad=[]\n        for index,request in active.items():\n            try:backend.validate(request);good.append(index)\n            except ValueError as exc:bad.append((index,exc))\n        for index,error in bad:\n            try:active[index]=generators[index].throw(error)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n        if not good:continue\n        responses=backend.generate([active[index] for index in good])\n        assert len(responses)==len(good)\n        for index,response in zip(good,responses,strict=True):\n            try:active[index]=generators[index].send(response)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n    return [completed[index] for index in range(len(generators))]\n\ndef prepare_t4_lora_kernel():\n    """Work around Triton 3.3 fp16 conversion failure on SM75; preserve FP32 accumulation."""\n    import torch,importlib.util,importlib.metadata,hashlib\n    path=Path(importlib.util.find_spec("vllm").origin).parent/"lora/ops/triton_ops/kernel_utils.py"\n    original="        accumulator += tl.dot(\\n            tiled_a,\\n            tiled_b,\\n        )"\n    patched="        # LifeMate SM75 compatibility: explicit exact fp16->fp32 before dot.\\n        accumulator += tl.dot(tiled_a.to(tl.float32), tiled_b.to(tl.float32), input_precision=\\"ieee\\")"\n    source=path.read_text()\n    capability=torch.cuda.get_device_capability(0)\n    if capability[0]>=8:\n        if patched in source:path.write_text(source.replace(patched,original))\n        return {"applied":False,"capability":list(capability)}\n    assert importlib.metadata.version("vllm")=="0.9.2"\n    assert importlib.metadata.version("triton")=="3.3.0"\n    assert original in source or patched in source,"Unexpected LoRA kernel source"\n    if original in source:path.write_text(source.replace(original,patched))\n    # Exercise the exact patched helper on the GPU against an FP32 PyTorch reference.\n    global triton,tl,mm_k\n    import triton\n    import triton.language as tl\n    from vllm.lora.ops.triton_ops.kernel_utils import mm_k\n    @triton.jit\n    def check_kernel(A,B,C,K:tl.constexpr):\n        rows=tl.arange(0,32)\n        cols=tl.arange(0,16)\n        kk=tl.arange(0,32)\n        a=A+rows[:,None]*K+kk[None,:]\n        b=B+kk[:,None]*16+cols[None,:]\n        acc=mm_k(a,b,1,16,kk,K,32,16,32,True,1,False,tl.float16)\n        tl.store(C+rows[:,None]*16+cols[None,:],acc)\n    errors=[]\n    torch.manual_seed(42)\n    for k in (128,2560):\n        a=torch.randn(32,k,device="cuda",dtype=torch.float16)\n        b=torch.randn(k,16,device="cuda",dtype=torch.float16)\n        c=torch.empty(32,16,device="cuda",dtype=torch.float32)\n        check_kernel[(1,)](a,b,c,k,num_warps=4)\n        ref=a.float()@b.float()\n        torch.testing.assert_close(c,ref,rtol=5e-4,atol=2e-3)\n        errors.append(float((c-ref).abs().max()))\n    return {"applied":True,"capability":list(capability),"operation":"LoRA dot with FP32 accumulation",\n            "reference_test":"pass","maximum_errors":errors,"kernel_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}\n\nclass Backend:\n    def __init__(self,model,revision,adapters,max_context=8192):\n        self.t4_compatibility=prepare_t4_lora_kernel()\n        print(dumps({"t4_compatibility":self.t4_compatibility}),flush=True)\n        from vllm import LLM,SamplingParams\n        from vllm.sampling_params import GuidedDecodingParams\n        from vllm.lora.request import LoRARequest\n        from transformers import AutoConfig,AutoModelForCausalLM\n        from safetensors import safe_open\n        import torch\n        assert torch.cuda.is_available()\n        assert torch.cuda.get_device_capability(0)>=(7,0)\n        self.max_context=max_context\n        self.requests={}\n        adapter_parameters=0\n        lineage_records={}\n        for index,role in enumerate(("global","planner","caller","summarizer"),start=1):\n            path=Path(adapters)/role\n            meta=json.loads((path/"lineage.json").read_text())\n            assert meta["stage"]==role and meta["model"]==model and meta["revision"]==revision\n            assert meta["final_adapter"]["finite"]\n            assert meta[\'prompt_sha256\']==digest(PROMPTS),\'Training/inference policy mismatch\'\n            assert meta[\'world_size\']==2 and meta[\'micro_batch\']==4 and meta[\'eval_batch_per_device\']==4\n            lineage_records[role]=meta\n            self.requests[role]=LoRARequest(role,index,str(path))\n            with safe_open(path/"adapter_model.safetensors",framework="pt",device="cpu") as handle:\n                adapter_parameters+=sum(math.prod(handle.get_slice(k).get_shape()) for k in handle.keys())\n        manifest_hashes={v[\'dataset_manifest_sha256\'] for v in lineage_records.values()}\n        assert len(manifest_hashes)==1\n        for role in (\'planner\',\'caller\',\'summarizer\'):\n            assert lineage_records[role][\'initial_adapter\'][\'sha256\']==lineage_records[\'global\'][\'final_adapter\'][\'sha256\']\n        self.dataset_manifest_sha256=next(iter(manifest_hashes))\n        config=AutoConfig.from_pretrained(model,revision=revision)\n        with torch.device("meta"):meta_model=AutoModelForCausalLM.from_config(config)\n        base_parameters=meta_model.num_parameters();del meta_model\n        self.parameter_report={"base":base_parameters,"comparison_adapters":adapter_parameters,\n                               "total_llm":base_parameters+adapter_parameters}\n        assert self.parameter_report["total_llm"]<5_000_000_000\n        self.llm=LLM(model=model,revision=revision,tokenizer=str(Path(adapters)/"global"),\n            dtype="float16",tensor_parallel_size=1,enable_lora=True,max_loras=4,max_cpu_loras=4,\n            max_lora_rank=16,max_model_len=max_context,max_num_seqs=4,max_num_batched_tokens=max_context,\n            gpu_memory_utilization=.90,enforce_eager=True,enable_prefix_caching=False,\n            enable_chunked_prefill=False,seed=42,trust_remote_code=False,\n            generation_config="vllm",disable_log_stats=True)\n        self.tokenizer=self.llm.get_tokenizer()\n        self.max_tokens={"global":768,"planner":320,"caller":512,"summarizer":384}\n        self.batch_sizes=[];self.roles_seen=Counter();self.schema_violations=0\n    def make_sampling(self,request):\n        from vllm import SamplingParams\n        from vllm.sampling_params import GuidedDecodingParams\n        # Fresh per request: never share mutable grammar state between batch members.\n        return SamplingParams(temperature=0,seed=42,\n            max_tokens=64 if request.get("probe") else self.max_tokens[request["role"]],\n            guided_decoding=GuidedDecodingParams(\n                json=request.get("schema",role_schema(request["role"])),backend="lm-format-enforcer"))\n    def validate(self,request):\n        role=request["role"]\n        prompt=self.tokenizer.apply_chat_template(request["messages"],tokenize=False,add_generation_prompt=True)\n        length=len(self.tokenizer.encode(prompt,add_special_tokens=False))\n        if length+(64 if request.get("probe") else self.max_tokens[role])>self.max_context:\n            raise ValueError("Context budget exceeded; no evidence is truncated")\n    def generate(self,requests):\n        assert 1<=len(requests)<=4\n        for r in requests:self.validate(r)\n        self.batch_sizes.append(len(requests));self.roles_seen.update(r["role"] for r in requests)\n        prompts=[self.tokenizer.apply_chat_template(r["messages"],tokenize=False,add_generation_prompt=True) for r in requests]\n        results=self.llm.generate(prompts,[self.make_sampling(r) for r in requests],\n            lora_request=[self.requests[r["role"]] for r in requests],use_tqdm=False)\n        from jsonschema import Draft202012Validator,ValidationError\n        responses=[]\n        for request,result in zip(requests,results,strict=True):\n            response={"raw":result.outputs[0].text,"finish_reason":result.outputs[0].finish_reason,\n                      "stop_reason":result.outputs[0].stop_reason,\n                      "prompt_sha256":digest(request["messages"]),"schema_sha256":digest(request.get("schema",role_schema(request["role"])))}\n            try:\n                value=parse_json(response["raw"])\n                Draft202012Validator(request.get("schema",role_schema(request["role"]))).validate(value)\n                response["schema_valid"]=True\n            except (ValueError,ValidationError) as exc:\n                response.update(schema_valid=False,schema_error=str(exc)[:300])\n                self.schema_violations+=1\n            responses.append(response)\n        return responses\n    def preflight(self):\n        probes=[]\n        for role in ("global","planner","caller","summarizer"):\n            schema={"type":"object","properties":{"probe":{"type":"string","enum":[role]}},\n                    "required":["probe"],"additionalProperties":False}\n            probes.append({"role":role,"probe":True,"schema":schema,"messages":[\n                {"role":"user","content":"Just say hello, no JSON."}]})\n        for request,result in zip(probes,self.generate(probes),strict=True):\n            assert result["schema_valid"] and parse_json(result["raw"])=={"probe":request["role"]},result\n        specs={"demo.inspect":{"name":"demo.inspect","kind":"read","description":"Read a synthetic probe record",\n            "parameters":{"type":"object","properties":{"token":{"type":"string","enum":["probe"]}},"required":["token"]}}}\n        row={"context":{},"history":[]};plan={"thought":"Cần đọc dữ liệu kiểm tra.","next":"caller","tool":"demo.inspect"}\n        requests=[]\n        for role in ("global","planner","caller","summarizer"):\n            trajectory=[] if role!=\'summarizer\' else [{\'action\':{\'name\':\'demo.inspect\',\'arguments\':{\'token\':\'probe\'}},\'observation\':{\'ok\':True,\'value\':\'probe\'}}]\n            requests.append({"role":role,"messages":build_messages(role,row,\'Đọc bản ghi kiểm tra được cung cấp.\',specs,trajectory,plan),\n                             \'schema\':request_schema(role,specs,plan)})\n        results=self.generate(requests)\n        assert all(r[\'schema_valid\'] for r in results),results\n        assert parse_json(results[2][\'raw\'])=={\'name\':\'demo.inspect\',\'arguments\':{\'token\':\'probe\'}}\n        contracts=[]\n        for name,parameters in [\n            (\'demo.integer\',{\'type\':\'object\',\'properties\':{\'count\':{\'type\':\'integer\',\'enum\':[2,4]}},\'required\':[\'count\']}),\n            (\'demo.alternative\',{\'type\':\'object\',\'properties\':{\'reference\':{\'type\':\'string\'},\'code\':{\'type\':\'integer\'}},\'required\':[],\'x-one-of\':[[\'reference\'],[\'code\']]}),\n            (\'demo.optional\',{\'type\':\'object\',\'properties\':{\'choice\':{\'type\':\'string\',\'enum\':[\'alpha\',\'beta\']}},\'required\':[]}),\n            (\'demo.flag\',{\'type\':\'object\',\'properties\':{\'enabled\':{\'type\':\'boolean\'}},\'required\':[\'enabled\']})]:\n            item={name:{\'name\':name,\'kind\':\'read\',\'description\':\'Synthetic tool contract probe\',\'parameters\':parameters}}\n            probe_plan={\'thought\':\'Kiểm tra schema giả lập.\',\'next\':\'caller\',\'tool\':name}\n            contracts.append({\'role\':\'caller\',\'schema\':request_schema(\'caller\',item,probe_plan),\n                \'messages\':build_messages(\'caller\',row,\'Thực hiện kiểm tra schema công cụ giả lập.\',item,[],probe_plan)})\n        results=self.generate(contracts)\n        assert all(r[\'schema_valid\'] for r in results),results\n        return {\'real_vllm_batch_size\':4,\'tested_prompt_roles\':list(self.requests),\'json_contracts\':\'pass\',\n            \'adversarial_schema_probe\':\'pass\',\'selected_tool_and_argument_schema_probes\':\'pass\',\n            \'probe_source\':\'synthetic fixtures only; no benchmark examples\'}\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in ("data-root","work","model","revision"):p.add_argument("--"+name,required=True)\n    p.add_argument("--adapters",required=True)\n    p.add_argument("--limit",type=int)\n    p.add_argument("--modes",nargs="+",choices=("single","multi"),default=["multi"])\n    p.add_argument("--max-calls",type=int,default=10)\n    p.add_argument("--max-decisions",type=int,default=16)\n    a=p.parse_args();root=Path(a.data_root);work=Path(a.work);data=work/"prepared"\n    manifest=json.loads((data/"manifest.json").read_text())\n    valid={key for key,value in manifest["split"].items() if value=="validation"}\n    source_rows=read_jsonl(root/"train/train.jsonl")\n    rows=[]\n    for raw in source_rows:\n        if raw[\'id\'] not in valid:continue\n        context={k:deepcopy(raw[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in raw[\'context\']}\n        context[\'installed_apps\']={app:{\'tools\':list(meta.get(\'tools\',[]))} for app,meta in raw[\'context\'][\'installed_apps\'].items()}\n        history=[]\n        for turn in raw.get(\'history\',[]):\n            keys=(\'role\',\'audio\') if turn[\'role\']==\'user\' else (\'role\',\'text\',\'tool_calls\',\'tool_results\')\n            history.append({k:deepcopy(turn[k]) for k in keys if k in turn})\n        rows.append({\'id\':raw[\'id\'],\'audio\':raw[\'audio\'],\'context\':context,\'history\':history})\n    del source_rows\n    assert len(rows)==len(valid)\n    if a.limit:rows=rows[:a.limit]\n    backend=Backend(a.model,a.revision,a.adapters)\n    import hashlib\n    assert backend.dataset_manifest_sha256==hashlib.sha256((data/"manifest.json").read_bytes()).hexdigest()\n    preflight=backend.preflight();write_json(work/"vllm_preflight.json",preflight)\n    print(dumps({"vllm_preflight":preflight}),flush=True)\n    transcripts=json.loads((work/"asr_transcripts.json").read_text())\n    asr_report=json.loads((work/"asr_report.json").read_text())\n    assert asr_report["validation_samples"]==len(rows) and not asr_report["oracle_fallback"]\n    for row in rows:\n        required=[row["audio"]]+[t["audio"] for t in row.get("history",[]) if t["role"]=="user"]\n        assert all(name in transcripts for name in required),"Missing current/history ASR transcript"\n    catalog=load_catalog(root,"train");ToolEnv=import_toolenv(root,"train")\n    from umilife.evaluate import evaluate\n    summary={}\n    for mode in a.modes:\n        name=f"validation_{mode}_finetuned_asr_vllm_batch4"\n        predictions=[];traces=[]\n        with (work/f"{name}.jsonl").open("w",encoding="utf-8") as pf,(work/f"{name}.traces.jsonl").open("w",encoding="utf-8") as tf:\n            for start in range(0,len(rows),4):\n                group=rows[start:start+4]\n                # One independent ToolEnv per sample, never shared across batch members.\n                envs=[ToolEnv("train",raw["id"],root=root) for raw in group]\n                safe_rows=[normalized_row(raw,transcript=transcripts) for raw in group]\n                gens=[rollout(row,transcripts[raw["audio"]],allowed_specs(row,catalog),env,mode,\n                              a.max_calls,a.max_decisions)\n                      for raw,row,env in zip(group,safe_rows,envs,strict=True)]\n                results=run_batch(gens,backend)\n                for prediction,trace in results:\n                    predictions.append(prediction);traces.append(trace)\n                    pf.write(dumps(prediction)+"\\n");pf.flush()\n                    tf.write(dumps(trace)+"\\n");tf.flush()\n                print(dumps({"mode":mode,"completed":len(predictions),"total":len(rows),\n                    "batch":len(group),"reasons":dict(Counter(t["reason"] for _,t in results))}),flush=True)\n        assert len(predictions)==len(rows) and {r["id"] for r in predictions}=={r["id"] for r in rows}\n        # Gold labels are first loaded by the scorer after all predictions in this mode exist.\n        references=json.loads((data/"references.json").read_text())\n        report=evaluate(predictions,references,catalog,traces)\n        report.update(time_limit_enforced=False,inference_engine="vllm",batch_size=4,\n                      metric_note="Quality-only local approximation; no time penalty.")\n        write_json(work/f"{name}.metrics.json",report)\n        summary[mode]={k:v for k,v in report.items() if k!="per_sample"}\n        print(dumps({"mode":mode,"metrics":summary[mode]}),flush=True)\n    assert backend.parameter_report["total_llm"]+asr_report["initializer_elements"]<6_000_000_000\n    write_json(work/\'prompt_policy.json\',{\'prompts\':PROMPTS,\'completion_review\':COMPLETION_REVIEW})\n    write_json(work/\'prompt_input_audit.json\',POLICY_AUDIT)\n    runtime={"prompt_policy":POLICY_AUDIT,"engine":"vllm","batch_size":4,"actual_batch_sizes":dict(Counter(backend.batch_sizes)),\n        "roles_seen":dict(backend.roles_seen),"dtype":"float16","base_quantization":None,\n        "training_quantization":"NF4","finetuned":True,"adapter_loaded":True,"execution_mode":"finetuned_asr_validation","parameters":backend.parameter_report,\n        "schema_enforced":True,"schema_violations":backend.schema_violations,\n        "t4_compatibility":backend.t4_compatibility,\n        "max_context":backend.max_context,"asr":True,"asr_model":"supplied_zipformer2","asr_report":asr_report,"time_limit_enforced":False,\n        "samples_per_mode":len(rows),"metrics":summary}\n    assert max(backend.batch_sizes)==4\n    write_json(work/"vllm_runtime.json",runtime)\nif __name__=="__main__":main()\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'generic-schema-and-completion-v2.1\'\nCOMMON=\'\'\'Bạn là trợ lý LifeMate tiếng Việt, thực hiện yêu cầu bằng công cụ đã cài.\nChỉ xử lý request hiện tại. History là các lượt trước: dùng để hiểu tham chiếu và lấy thông tin đã biết, nhưng không coi hành động trong history là đã hoàn thành request mới. Chỉ trajectory hiện tại chứng minh các bước vừa thực hiện.\nRequest nguyên văn là nguồn chính: giữ đủ từ hạn định đối tượng, số lượng, đơn vị và phủ định; không thay bằng một bản diễn giải mất chi tiết trong thought.\nĐọc toàn bộ ý định, lời sửa đổi, điều kiện và nhiều việc trong cùng câu. Văn bản đến từ nhận dạng tiếng nói: diễn giải theo ngữ cảnh, đơn vị và danh mục công cụ; không tự bịa tên riêng, ID hoặc dữ kiện.\nƯu tiên thông tin có sẵn trong context, history, trajectory. Khi thiếu ID hay dữ liệu, dùng công cụ tra cứu phù hợp; x-ref mô tả nguồn định danh. Không chọn ngẫu nhiên khi kết quả có nhiều đối tượng. Nếu tham số bắt buộc không thể tra cứu hoặc suy ra chắc chắn thì hỏi qua công cụ hệ thống báo thiếu thông tin.\nHiểu số và ngày giờ từ request, dùng current_time và timezone của context. Giờ tương đối phải tính đúng ngày; biểu thức giờ kém phút là giờ tròn trừ số phút, sau khi xác định buổi. Giữ đơn vị và kiểm tra phép tính trước khi tạo arguments.\nVới điều kiện, đọc dữ liệu để xác định nhánh rồi mới thực hiện thao tác phụ thuộc. Không thêm bước đọc khi thông tin cần thiết đã có. Tránh lặp cùng lời gọi khi chưa có dữ kiện mới, tránh thực hiện lại thao tác ghi đã thành công nếu người dùng không yêu cầu lặp.\nMột observation có ok=true nhưng done=false vẫn chưa thực hiện được thao tác. Kết quả tra cứu rỗng không chứng minh yêu cầu ngoài phạm vi: cân nhắc điều chỉnh bộ lọc, tra cứu rộng hơn theo schema hoặc hỏi thông tin còn thiếu. Không tưởng tượng kết quả.\nChỉ xuất một JSON đúng hợp đồng đầu ra, không markdown, không văn bản ngoài JSON.\'\'\'\n\nPROMPTS={\n \'planner\':COMMON+\'\'\'\nBạn là Planner. Chọn đúng một bước tiếp theo theo ý định và những gì thực sự còn thiếu.\nThought chỉ một hoặc hai câu ngắn về bước còn thiếu, bằng chứng đã có và lý do chọn bước này; không kể lại toàn bộ request. Khi next=caller, tool phải là tên chính xác có trong tools. Xem kind, parameters, required và x-ref để biết cách lấy dữ liệu phụ thuộc trước khi ghi.\nChỉ chọn summarizer khi mọi ý của request đã được thực hiện có bằng chứng, điều kiện không cần thực hiện đã được kiểm tra, người dùng không yêu cầu thao tác, hoặc đã gọi công cụ hỏi bổ sung và cần chờ trả lời. Tìm được thông tin chưa đồng nghĩa đã thực hiện hành động người dùng muốn.\nChỉ chọn give_up khi yêu cầu ngoài khả năng của toàn bộ công cụ đã cài. Thiếu một ID, tra cứu rỗng hoặc chưa biết cách dùng tool không tự động có nghĩa ngoài phạm vi.\nJSON gồm thought và next. next là caller, summarizer hoặc give_up. Chỉ khi next=caller mới thêm tool.\'\'\',\n \'caller\':COMMON+\'\'\'\nBạn là Caller. Tạo đúng một lời gọi cho tool Planner đã chọn. Output có đúng hai trường name và arguments; name phải khớp tên trong selected_tool, arguments tuân thủ schema được cung cấp.\nĐọc kỹ ý định, plan và observation để điền giá trị. Dùng đúng kiểu số nguyên/số thực/chuỗi/mảng và đúng enum. Không bọc lời gọi vào một tool khác. Không tạo trường ngoài properties. Tham số tùy chọn không có căn cứ thì bỏ qua; không thêm bộ lọc phỏng đoán. Nhóm x-one-of chỉ cần đủ một phương án có dữ kiện.\nPlan chỉ chọn bước và tool. Luôn đối chiếu giá trị và đối tượng với request nguyên văn; không sao chép một số hay tên sai từ thought. Giữ đủ từ hạn định của tên cần tìm khi dùng bộ lọc.\nVới nội dung gửi hoặc lưu, giữ đúng nội dung người dùng yêu cầu, không tự viết thêm lời bình luận. Không khẳng định đã gọi công cụ: chương trình sẽ thực hiện sau khi bạn trả JSON.\'\'\',\n \'summarizer\':COMMON+\'\'\'\nBạn là Summarizer. Viết answer ngắn, tự nhiên và trung thực từ observation của trajectory hiện tại. Không gọi thêm tool và không chép toàn bộ JSON kết quả.\nChỉ báo một thao tác thành công nếu đúng tool đã thực sự chạy và observation xác nhận. Nếu chưa thực hiện thì nói rõ; nếu đã hỏi bổ sung thì nêu thông tin còn cần và chờ người dùng. Không lấy lời nói của assistant trong history làm bằng chứng thực hiện request mới.\nOutput chỉ có answer.\'\'\',\n \'global\':COMMON+\'\'\'\nBạn xử lý từng bước theo ReAct. Output thought và next. next=caller cần tool và action gồm name, arguments đúng schema; next=summarizer cần answer ngắn dựa trên kết quả thực tế; next=give_up chỉ khi ngoài khả năng toàn bộ tool đã cài.\nĐối chiếu toàn bộ mục tiêu với trajectory trước khi kết thúc. Dữ liệu đã tìm thấy chưa có nghĩa thao tác yêu cầu đã thực hiện.\'\'\'\n}\n\nCOMPLETION_REVIEW=\'\'\'Kiểm tra lại đề xuất kết thúc trước khi trả lời. Đây chưa phải thao tác thực thi.\nĐối chiếu TỪNG ý của request với trajectory hiện tại. Có việc chỉ mới được tra thông tin nhưng chưa thực hiện không? Có đang nhầm history với request hiện tại không? Có observation done=false hoặc kết quả rỗng chưa được xử lý không?\nNếu còn việc có thể thực hiện, chọn caller và tool kế tiếp. Nếu thiếu thông tin thì ưu tiên tra cứu, chỉ hỏi bổ sung khi không thể tìm. Chỉ summarizer khi đã đủ bằng chứng hoàn thành/không cần thao tác/đang chờ người dùng; give_up chỉ khi ngoài khả năng tool đã cài.\'\'\'\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':480 if global_mode else 240};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools\']=compact_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool=deepcopy(specs[selected]))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools\']=compact_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Tool cần bổ sung không có trong danh mục đã cài\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params phải là tên tham số có trong schema của tool_name, không phải câu hỏi tự do\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/transcribe_supplied.py': '"""Recognize current/history audio using only the supplied Zipformer2 ONNX files."""\nimport argparse,hashlib,json,math,time\nfrom pathlib import Path\nimport numpy as np\nimport soundfile as sf\nfrom scipy.signal import resample_poly\nimport sherpa_onnx\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef transcribe(root,work,model,limit=None,threads=4):\n    root=Path(root);work=Path(work);model=Path(model)\n    manifest=json.loads((model/\'model_manifest.json\').read_text())\n    assert manifest[\'model_type\']==\'zipformer2\' and not manifest[\'streaming\']\n    for name,info in manifest[\'files\'].items():assert digest(model/name)==info[\'sha256\'],f\'Wrong ASR file: {name}\'\n    fingerprint=digest(model/\'model_manifest.json\')\n    valid=set((work/\'split.txt\').read_text().splitlines());assert 0<len(valid)<=100\n    rows=[json.loads(line) for line in (root/\'train/train.jsonl\').open()]\n    selected=[r for r in rows if r[\'id\'] in valid];assert len(selected)==len(valid)\n    if limit:selected=selected[:limit]\n    audio=[]\n    for row in selected:\n        audio.append(row[\'audio\'])\n        audio.extend(t[\'audio\'] for t in row.get(\'history\',[]) if t[\'role\']==\'user\')\n    audio=list(dict.fromkeys(audio))\n    config={\'model_manifest_sha256\':fingerprint,\'split_sha256\':digest(work/\'split.txt\'),\n        \'selected_ids\':[r[\'id\'] for r in selected],\'sample_rate\':16000,\'decoding_method\':\'greedy_search\',\n        \'provider\':\'cpu\',\'sherpa_onnx\':sherpa_onnx.__version__,\'source\':\'supplied_asr_model\'}\n    config_path=work/\'asr_config.json\'\n    if config_path.exists():assert json.loads(config_path.read_text())==config,\'Stale ASR cache configuration\'\n    else:config_path.write_text(json.dumps(config,indent=2))\n    cache_path=work/\'asr_transcripts.jsonl\';cache={}\n    if cache_path.exists():\n        for line in cache_path.read_text().splitlines():\n            record=json.loads(line);assert record[\'model_manifest_sha256\']==fingerprint\n            cache[record[\'audio\']]=record\n    recognizer=sherpa_onnx.OfflineRecognizer.from_transducer(\n        encoder=str(model/\'encoder.onnx\'),decoder=str(model/\'decoder.onnx\'),joiner=str(model/\'joiner.onnx\'),\n        tokens=str(model/\'tokens.txt\'),num_threads=threads,sample_rate=16000,feature_dim=80,\n        decoding_method=\'greedy_search\',provider=\'cpu\')\n    started=time.monotonic()\n    with cache_path.open(\'a\',encoding=\'utf-8\') as output:\n        for index,name in enumerate(audio,1):\n            path=(root/\'train\'/name).resolve();assert path.is_relative_to((root/\'train\').resolve())\n            audio_sha=digest(path)\n            if name in cache:\n                assert cache[name][\'audio_sha256\']==audio_sha,\'Audio changed after transcription\'\n                continue\n            tick=time.monotonic()\n            try:\n                samples,sample_rate=sf.read(path,dtype=\'float32\',always_2d=True)\n                samples=samples.mean(axis=1)\n                assert len(samples)>0 and np.isfinite(samples).all(),\'Empty or non-finite waveform\'\n                duration=len(samples)/sample_rate\n                if sample_rate!=16000:\n                    factor=math.gcd(sample_rate,16000)\n                    samples=resample_poly(samples,16000//factor,sample_rate//factor)\n                stream=recognizer.create_stream()\n                stream.accept_waveform(16000,np.ascontiguousarray(samples,dtype=np.float32))\n                recognizer.decode_stream(stream)\n                record={\'audio\':name,\'text\':stream.result.text.strip(),\'audio_sha256\':audio_sha,\n                    \'model_manifest_sha256\':fingerprint,\'elapsed_s\':time.monotonic()-tick,\'duration_s\':duration,\n                    \'source\':\'supplied_asr_model\',\'provider\':\'cpu\'}\n            except Exception as exc:\n                (work/\'asr_error.json\').write_text(json.dumps({\'audio\':name,\'error\':str(exc)}))\n                raise\n            output.write(json.dumps(record,ensure_ascii=False)+\'\\n\');output.flush();cache[name]=record\n            print(json.dumps({\'stage\':\'asr\',\'completed\':index,\'total_audio\':len(audio),\n                \'validation_samples\':len(selected),\'audio\':name,\'elapsed_s\':record[\'elapsed_s\']},ensure_ascii=False),flush=True)\n    assert set(audio)<=cache.keys()\n    transcripts={name:cache[name][\'text\'] for name in audio}\n    (work/\'asr_transcripts.json\').write_text(json.dumps(transcripts,ensure_ascii=False,indent=2))\n    report={\'status\':\'complete\',\'validation_samples\':len(selected),\'unique_audio\':len(audio),\n        \'current_and_history_audio\':True,\'oracle_fallback\':False,\'training\':False,\n        \'empty_transcripts\':sum(not value.strip() for value in transcripts.values()),\n        \'wall_s_this_run\':time.monotonic()-started,\n        \'audio_duration_s\':sum(cache[name][\'duration_s\'] for name in audio),\n        \'initializer_elements\':sum(info.get(\'initializer_elements\',0) for info in manifest[\'files\'].values()),\n        \'model_manifest_sha256\':fingerprint,\'provider\':\'cpu\',\'model_type\':\'zipformer2\'}\n    (work/\'asr_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'ASR_COMPLETE\',json.dumps(report),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model-dir\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--limit\',type=int);p.add_argument(\'--threads\',type=int,default=4)\n    a=p.parse_args();transcribe(a.data_root,a.work,a.model_dir,a.limit,a.threads)\n', 'requirements_train_2gpu.lock': 'accelerate==1.10.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\nagent-detector==2.0.0\n    # via fastapi-cloud-cli\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   fsspec\n    #   vllm\naiosignal==1.4.0\n    # via aiohttp\nannotated-doc==0.0.5\n    # via\n    #   fastapi\n    #   typer\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\n    #   openai\n    #   starlette\n    #   watchfiles\nastor==0.8.1\n    # via depyf\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   jsonschema\n    #   referencing\nbitsandbytes==0.47.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nblake3==1.0.11\n    # via vllm\ncachetools==7.2.8\n    # via vllm\ncbor2==6.2.0.post2\n    # via vllm\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\n    #   sentry-sdk\ncffi==2.1.1\n    # via soundfile\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via\n    #   ray\n    #   rich-toolkit\n    #   uvicorn\ncloudpickle==3.1.2\n    # via vllm\ncompressed-tensors==0.11.0\n    # via vllm\ncuda-pathfinder==1.8.5\n    # via cupy-cuda12x\ncupy-cuda12x==14.2.0\n    # via ray\ncut-cross-entropy==25.1.1\n    # via unsloth-zoo\ndatasets==3.6.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\ndepyf==0.19.0\n    # via vllm\ndetect-installer==0.2.1\n    # via fastapi-cloud-cli\ndiffusers==0.35.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\ndill==0.3.8\n    # via\n    #   datasets\n    #   depyf\n    #   multiprocess\ndiskcache==5.6.3\n    # via vllm\ndnspython==2.9.1\n    # via email-validator\ndocstring-parser==0.18.0\n    # via tyro\neinops==0.8.2\n    # via vllm\nemail-validator==2.3.0\n    # via\n    #   fastapi\n    #   pydantic\nfastapi==0.162.0\n    # via vllm\nfastapi-cli==0.0.32\n    # via fastapi\nfastapi-cloud-cli==0.26.0\n    # via fastapi-cli\nfastar==0.12.0\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nfilelock==4.2.7\n    # via\n    #   datasets\n    #   diffusers\n    #   huggingface-hub\n    #   ray\n    #   torch\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nfrozendict==2.5.1\n    # via compressed-tensors\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2025.3.0\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\ngguf==0.19.0\n    # via vllm\ngoogleapis-common-protos==1.76.0\n    # via opentelemetry-exporter-otlp-proto-http\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nhf-transfer==0.1.9\n    # via\n    #   unsloth\n    #   unsloth-zoo\nhf-xet==1.8.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttptools==0.9.0\n    # via uvicorn\nhttpx==0.28.1\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nhttpx2==2.16.0\n    # via\n    #   fastapi\n    #   openai\nhuggingface-hub==0.35.3\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   datasets\n    #   diffusers\n    #   peft\n    #   tokenizers\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimportlib-metadata==10.1.0\n    # via diffusers\ninteregular==0.3.3\n    # via lm-format-enforcer\njinja2==3.1.6\n    # via\n    #   fastapi\n    #   torch\njiter==0.18.0\n    # via openai\njsonschema==4.26.0\n    # via\n    #   mistral-common\n    #   ray\njsonschema-specifications==2025.9.1\n    # via jsonschema\nlark==1.2.2\n    # via vllm\nllguidance==0.7.30\n    # via vllm\nllvmlite==0.44.0\n    # via numba\nlm-format-enforcer==0.11.3\n    # via vllm\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmdurl==0.1.2\n    # via markdown-it-py\nmistral-common==1.13.1\n    # via vllm\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via ray\nmsgspec==0.22.0\n    # via\n    #   unsloth-zoo\n    #   vllm\nmultidict==7.2.1\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.16\n    # via datasets\nnetworkx==3.7\n    # via torch\nninja==1.13.2\n    # via\n    #   vllm\n    #   xgrammar\nnumba==0.61.2\n    # via vllm\nnumpy==2.2.6\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   cupy-cuda12x\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   mistral-common\n    #   numba\n    #   opencv-python-headless\n    #   pandas\n    #   peft\n    #   scipy\n    #   soundfile\n    #   soxr\n    #   torchvision\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.3\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nopenai==3.42.1\n    # via vllm\nopenai-harmony==0.0.8\n    # via vllm\nopencv-python-headless==5.0.0.93\n    # via\n    #   mistral-common\n    #   vllm\nopentelemetry-api==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\nopentelemetry-exporter-http-transport==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-common==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-common==1.45.2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-http==1.45.2\n    # via fastapi\nopentelemetry-proto==1.45.2\n    # via\n    #   opentelemetry-exporter-otlp-proto-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-sdk==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-otlp-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-semantic-conventions==0.66b2\n    # via opentelemetry-sdk\noutlines-core==0.2.11\n    # via vllm\npackaging==26.3\n    # via\n    #   accelerate\n    #   datasets\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   wheel\npandas==3.1.0\n    # via datasets\npartial-json-parser==0.2.1.1.post7\n    # via vllm\npeft==0.17.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\npillow==13.0.0\n    # via\n    #   diffusers\n    #   mistral-common\n    #   torchvision\n    #   unsloth-zoo\n    #   vllm\nprometheus-client==0.26.0\n    # via\n    #   prometheus-fastapi-instrumentator\n    #   vllm\nprometheus-fastapi-instrumentator==8.1.0\n    # via vllm\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nprotobuf==7.36.2\n    # via\n    #   googleapis-common-protos\n    #   opentelemetry-proto\n    #   ray\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npsutil==7.2.2\n    # via\n    #   accelerate\n    #   peft\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npy-cpuinfo==9.0.0\n    # via vllm\npyarrow==26.0.0\n    # via datasets\npybase64==1.5.1\n    # via vllm\npycountry==26.2.16\n    # via pydantic-extra-types\npycparser==3.12\n    # via cffi\npydantic==2.14.2\n    # via\n    #   compressed-tensors\n    #   fastapi\n    #   fastapi-cloud-cli\n    #   lm-format-enforcer\n    #   mistral-common\n    #   openai\n    #   openai-harmony\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   vllm\n    #   xgrammar\npydantic-core==2.50.2\n    # via pydantic\npydantic-extra-types==2.11.1\n    # via\n    #   fastapi\n    #   mistral-common\npydantic-settings==2.16.2\n    # via fastapi\npygments==2.21.0\n    # via rich\npython-dateutil==4.0.0\n    # via pandas\npython-dotenv==1.2.4\n    # via\n    #   pydantic-settings\n    #   uvicorn\npython-json-logger==4.2.0\n    # via vllm\npython-multipart==0.0.33\n    # via fastapi\npyyaml==6.0.3\n    # via\n    #   accelerate\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   uvicorn\n    #   vllm\npyzmq==27.2.0\n    # via vllm\nray==2.59.0\n    # via vllm\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-specifications\nregex==2026.9.29\n    # via\n    #   diffusers\n    #   tiktoken\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nrequests==2.34.2\n    # via\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   huggingface-hub\n    #   mistral-common\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   ray\n    #   tiktoken\n    #   transformers\n    #   vllm\nrich==15.0.0\n    # via\n    #   rich-toolkit\n    #   typer\nrich-toolkit==0.20.6\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nrignore==0.8.1\n    # via fastapi-cloud-cli\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nsafetensors==0.8.0\n    # via\n    #   accelerate\n    #   diffusers\n    #   peft\n    #   transformers\nscipy==1.18.1\n    # via vllm\nsentencepiece==0.2.2\n    # via\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\nsentry-sdk==2.73.0\n    # via fastapi-cloud-cli\nsetproctitle==1.3.8\n    # via vllm\nsetuptools==79.0.1\n    # via\n    #   torch\n    #   triton\n    #   vllm\nshellingham==1.5.4\n    # via typer\nsix==1.17.0\n    # via vllm\nsniffio==1.3.1\n    # via openai\nsoundfile==0.14.0\n    # via mistral-common\nsoxr==1.1.0\n    # via mistral-common\nstarlette==1.10.0\n    # via\n    #   fastapi\n    #   prometheus-fastapi-instrumentator\nsympy==1.14.0\n    # via torch\ntiktoken==0.14.0\n    # via\n    #   mistral-common\n    #   vllm\ntokenizers==0.22.2\n    # via\n    #   transformers\n    #   vllm\ntorch==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   compressed-tensors\n    #   cut-cross-entropy\n    #   peft\n    #   torchaudio\n    #   torchvision\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\ntorchao==0.13.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth-zoo\ntorchaudio==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   vllm\ntorchvision==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   peft\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\ntransformers==4.56.2\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   compressed-tensors\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntriton==3.4.0\n    # via\n    #   cut-cross-entropy\n    #   torch\n    #   unsloth\n    #   unsloth-zoo\n    #   xgrammar\ntrl==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via tyro\ntyper==0.27.3\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   anyio\n    #   fastapi\n    #   httpx2\n    #   huggingface-hub\n    #   openai\n    #   opentelemetry-api\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   referencing\n    #   rich-toolkit\n    #   soundfile\n    #   starlette\n    #   torch\n    #   typeguard\n    #   typing-inspection\n    #   tyro\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntyping-inspection==0.4.4\n    # via\n    #   fastapi\n    #   pydantic\n    #   pydantic-settings\ntyro==1.0.16\n    # via\n    #   unsloth\n    #   unsloth-zoo\ntzdata==2026.6\n    # via python-dateutil\nunsloth==2025.10.9\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nunsloth-zoo==2025.10.10\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nurllib3==2.8.0\n    # via\n    #   requests\n    #   sentry-sdk\nuvicorn==0.54.0\n    # via\n    #   fastapi\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nuvloop==0.23.0\n    # via uvicorn\nvllm==0.10.2\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nwatchfiles==1.3.0\n    # via\n    #   uvicorn\n    #   vllm\nwebsockets==17.4\n    # via uvicorn\nwheel==0.48.0\n    # via\n    #   unsloth\n    #   unsloth-zoo\nwrapt==1.17.3\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nxformers==0.0.32.post1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\nxgrammar==0.1.23\n    # via vllm\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.2\n    # via importlib-metadata\n', 'requirements_infer_2gpu.lock': '--extra-index-url https://download.pytorch.org/whl/cu128\naccelerate==1.15.0\nagent-detector==2.0.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nairportsdata==20261030\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\nastor==0.8.1\nattrs==26.1.0\nblake3==1.0.11\ncachetools==7.2.8\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.10.2\ncuda-pathfinder==1.8.5\ncupy-cuda12x==14.2.0\ndepyf==0.18.0\ndetect-installer==0.2.1\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.9.1\neinops==0.8.2\nemail-validator==2.3.0\nfastapi==0.162.0\nfastapi-cli==0.0.32\nfastapi-cloud-cli==0.26.0\nfastar==0.12.0\nfilelock==4.2.7\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngguf==0.19.0\ngoogleapis-common-protos==1.76.0\nh11==0.16.0\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpcore2==2.16.0\nhttptools==0.9.0\nhttpx==0.28.1\nhttpx2==2.16.0\nhuggingface-hub==0.35.3\nidna==3.20\ninteregular==0.3.3\njinja2==3.1.6\njiter==0.18.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nlark==1.2.2\nllguidance==0.7.30\nllvmlite==0.44.0\nlm-format-enforcer==0.10.12\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmistral-common==1.13.1\nmpmath==1.3.0\nmsgpack==1.2.4\nmsgspec==0.22.0\nmultidict==7.2.1\nnest-asyncio==1.6.0\nnetworkx==3.7\nninja==1.13.2\nnumba==0.61.2\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.3.14\nnvidia-cuda-cupti-cu12==12.8.57\nnvidia-cuda-nvrtc-cu12==12.8.61\nnvidia-cuda-runtime-cu12==12.8.57\nnvidia-cudnn-cu12==9.7.1.26\nnvidia-cufft-cu12==11.3.3.41\nnvidia-cufile-cu12==1.13.0.11\nnvidia-curand-cu12==10.3.9.55\nnvidia-cusolver-cu12==11.7.2.55\nnvidia-cusparse-cu12==12.5.7.53\nnvidia-cusparselt-cu12==0.6.3\nnvidia-nccl-cu12==2.26.2\nnvidia-nvjitlink-cu12==12.8.61\nnvidia-nvtx-cu12==12.8.55\nopenai==1.90.0\nopencv-python-headless==5.0.0.93\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noutlines==0.1.11\noutlines-core==0.1.26\npackaging==26.3\npartial-json-parser==0.2.1.1.post7\npeft==0.17.1\npillow==13.0.0\nprometheus-client==0.26.0\nprometheus-fastapi-instrumentator==8.1.0\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy-cpuinfo==9.0.0\npybase64==1.5.1\npycountry==26.2.16\npydantic==2.14.2\npydantic-core==2.50.2\npydantic-extra-types==2.11.1\npydantic-settings==2.16.2\npygments==2.21.0\npython-dotenv==1.2.4\npython-json-logger==4.2.0\npython-multipart==0.0.33\npyyaml==6.0.3\npyzmq==27.2.0\nray==2.59.0\nreferencing==0.37.0\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nrich-toolkit==0.20.6\nrignore==0.8.1\nrpds-py==2026.9.1\nsafetensors==0.8.0\nscipy==1.18.1\nsentencepiece==0.2.2\nsentry-sdk==2.73.0\nsetuptools==79.0.1\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nstarlette==1.10.0\nsympy==1.14.0\ntiktoken==0.14.0\ntokenizers==0.21.4\ntorch==2.7.0+cu128\ntorchaudio==2.7.0+cu128\ntorchvision==0.22.0+cu128\ntqdm==4.70.1\ntransformers==4.53.3\ntriton==3.3.0\ntruststore==0.10.5\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\nuvloop==0.23.0\nvllm==0.9.2\nwatchfiles==1.3.0\nwebsockets==17.4\nwrapt==1.17.3\nxformers==0.0.30\nxgrammar==0.1.19\nyarl==1.25.1\n', 'requirements_asr_2gpu.lock': 'cffi==2.1.1\nnumpy==2.2.6\npycparser==3.12\nscipy==1.16.2\nsherpa-onnx==2.0.1\nsherpa-onnx-core==2.0.1\nsoundfile==0.13.1\n', 'split.txt': 'train_1474\ntrain_0255\ntrain_0063\ntrain_1715\ntrain_0618\ntrain_0538\ntrain_0486\ntrain_0313\ntrain_1703\ntrain_0236\ntrain_1560\ntrain_1713\ntrain_1249\ntrain_0203\ntrain_1356\ntrain_0959\ntrain_0080\ntrain_0075\ntrain_0216\ntrain_0476\ntrain_0508\ntrain_1155\ntrain_1379\ntrain_0066\ntrain_1289\ntrain_0435\ntrain_1653\ntrain_1497\ntrain_1617\ntrain_0953\ntrain_0480\ntrain_1021\ntrain_1353\ntrain_0624\ntrain_0015\ntrain_0354\ntrain_1608\ntrain_0960\ntrain_0779\ntrain_0346\ntrain_0469\ntrain_0772\ntrain_0214\ntrain_0864\ntrain_0225\ntrain_0819\ntrain_0787\ntrain_1384\ntrain_0596\ntrain_0106\ntrain_1623\ntrain_0923\ntrain_1300\ntrain_1486\ntrain_1352\ntrain_1355\ntrain_1014\ntrain_1380\ntrain_1542\ntrain_1295\ntrain_0943\ntrain_1070\ntrain_1705\ntrain_1448\ntrain_0958\ntrain_1081\ntrain_1137\ntrain_1248\ntrain_1478\ntrain_1305\ntrain_0017\ntrain_0977\ntrain_0933\ntrain_1016\ntrain_1128\ntrain_1460\ntrain_1124\ntrain_1658\ntrain_1265\ntrain_1491\ntrain_1432\ntrain_1278\ntrain_1405\ntrain_1546\ntrain_0867\ntrain_1117\ntrain_1654\ntrain_0505\ntrain_1118\ntrain_0572\ntrain_1148\ntrain_1433\ntrain_1556\ntrain_1557\ntrain_1615\ntrain_1317\ntrain_1215\ntrain_1346\ntrain_1440\ntrain_0536\n'}
RUNTIME_SHA='258897097e18252ce0b08aa2492515715ad36e64df07a81ea44fa0403fb81d77'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==RUNTIME_SHA
for name,content in PAYLOAD.items():
    target=SOURCE/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_text(content,encoding='utf-8')
SPLIT_FILE=WORK/'split.txt';SPLIT_FILE.write_text(PAYLOAD['split.txt'],encoding='utf-8')
assert hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()=='631f8cfab61ddde833477f743cf1a65a9012526e0178e97046a36ee864e3d505'
RUN_CONFIG={'model':MODEL,'revision':MODEL_REVISION,'source_core':GIT_REF,'runtime_sha256':RUNTIME_SHA,
    'world_size':2,'train_micro_batch_per_gpu':4,'validation_loss_batch_per_gpu':4,
    'gradient_accumulation':1,'effective_train_batch':8,'inference_batch':4,'epochs':EPOCHS,
    'valid_limit':VALID_LIMIT,'modes':list(MODES),'asr_finetune':False}
config=WORK/'run_config.json'
if config.exists():assert json.loads(config.read_text())==RUN_CONFIG,'RUN_NAME already has different settings'
else:config.write_text(json.dumps(RUN_CONFIG,indent=2))
print('Runtime:',RUNTIME_SHA,'Outputs:',WORK)


In [ ]:
sys.path.insert(0, str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT, archive_path = find_data_root("/kaggle/input")
if DATA_ROOT is None:
    if archive_path is None:
        raise FileNotFoundError("Attach cymeu4l0t/zaloai-dataset.")
    destination = Path("/tmp/competition_data_codex")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive member")
        archive.extractall(destination)
    DATA_ROOT, _ = find_data_root(destination)
assert DATA_ROOT is not None, "Competition train manifests not found"
TRAIN_SHA = "24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8"
assert hashlib.sha256((DATA_ROOT/"train/train.jsonl").read_bytes()).hexdigest() == TRAIN_SHA

ASR_MODEL_DIR=next((p for p in [Path("/kaggle/input/lifemate-supplied-zipformer-asr"),
    Path("/kaggle/input/datasets/cymeu4l0t/lifemate-supplied-zipformer-asr")]
    if (p/"model_manifest.json").is_file()),None)
assert ASR_MODEL_DIR is not None,"Attach cymeu4l0t/lifemate-supplied-zipformer-asr"
ASR_MODEL_SHA='c1077ee90c0933d76329af5ac4f85eabd5c177dcf94c5bf5adb004dcf5051c9b'
assert hashlib.sha256((ASR_MODEL_DIR/"model_manifest.json").read_bytes()).hexdigest()==ASR_MODEL_SHA

RATIONALE_PATH=next((p for p in [Path('/kaggle/input/lifemate-codex-workflow-rationales/rationales.jsonl'),
    Path('/kaggle/input/datasets/cymeu4l0t/lifemate-codex-workflow-rationales/rationales.jsonl')] if p.is_file()),None)
assert RATIONALE_PATH is not None,'Attach cymeu4l0t/lifemate-codex-workflow-rationales'
assert hashlib.sha256(RATIONALE_PATH.read_bytes()).hexdigest()=='d14768df60e155521a57998a5c6d0c4c1cc3b6372b74cce7a955098fd3fdf497'


In [ ]:
# Separate Torch/CUDA runtimes avoid Unsloth/vLLM package conflicts.
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
def runtime(name,lock,gpus):
    path=Path('/tmp')/name
    if not (path/'bin/python').exists():
        subprocess.run([*UV,'venv','--python','3.12','--python-preference','only-managed','--seed',str(path)],check=True)
    python=str(path/'bin/python')
    subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',python,str(SOURCE/lock)],check=True)
    subprocess.run([*UV,'pip','check','--python',python],check=True)
    purelib=Path(subprocess.check_output([python,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
    env=os.environ.copy()
    for key in ('PYTHONPATH','PYTHONHOME'):env.pop(key,None)
    env.update(CUDA_VISIBLE_DEVICES=gpus,PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',
        HF_HOME='/tmp/lifemate_hf_cache',UNSLOTH_COMPILE_DISABLE='1',OMP_NUM_THREADS='2',
        VLLM_USE_V1='0',VLLM_ATTENTION_BACKEND='XFORMERS',VLLM_WORKER_MULTIPROC_METHOD='spawn',
        VLLM_USAGE_STATS_ENABLED='0',DO_NOT_TRACK='1',TORCH_NCCL_ASYNC_ERROR_HANDLING='1')
    env['LD_LIBRARY_PATH']=':'.join([str(p) for p in sorted((purelib/'nvidia').glob('*/lib'))]+[os.environ.get('LD_LIBRARY_PATH','')])
    return python,env
TRAIN_PYTHON,TRAIN_ENV=runtime('lifemate_unsloth_training_env','requirements_train_2gpu.lock','0,1')
def train_module(module,args,distributed=False):
    command=[TRAIN_PYTHON,'-u']
    if distributed:command+=['-m','torch.distributed.run','--standalone','--nproc_per_node=2','--module',module]
    else:command+=['-m',module]
    subprocess.run([*command,*map(str,args)],cwd=SOURCE,env=TRAIN_ENV,check=True)
INFER_PYTHON,INFER_ENV=runtime('lifemate_vllm092_env','requirements_infer_2gpu.lock','0')
ASR_PYTHON,ASR_ENV=runtime('lifemate_supplied_asr_env','requirements_asr_2gpu.lock','')
ASR_ENV['OMP_NUM_THREADS']=str(ASR_THREADS)
subprocess.run([INFER_PYTHON,'-c',"import vllm,torch;from vllm.sampling_params import GuidedDecodingParams;from vllm.lora.request import LoRARequest;assert vllm.__version__=='0.9.2';assert torch.cuda.is_available();print(vllm.__version__,torch.__version__)"],cwd=SOURCE,env=INFER_ENV,check=True)
subprocess.run([ASR_PYTHON,'-c',"import sherpa_onnx,soundfile,scipy;print('ASR imports pass')"],cwd=SOURCE,env=ASR_ENV,check=True)
train_module('umilife.preflight_2gpu',[],distributed=True)


In [ ]:
train_module('umilife.prepare_2gpu',['--data-root',DATA_ROOT,'--output',DATA,
    '--rationales',RATIONALE_PATH,'--split-file',SPLIT_FILE,'--model',MODEL,'--revision',MODEL_REVISION])
manifest=json.loads((DATA/'manifest.json').read_text())
assert manifest['train_episodes']==1445 and manifest['validation_episodes']==100
print(json.dumps(manifest['counts'],indent=2));print(json.dumps(manifest['tokenization'],indent=2))


## Global → local fine-tuning

Mỗi stage chạy hai process DDP; mỗi process giữ một bản QLoRA trên GPU riêng. Validation loss được cộng theo số token assistant trên cả hai GPU, không lặp mẫu valid. Tất cả rank đồng bộ trước/sau khi rank 0 lưu checkpoint; mọi rank cùng nạp adapter tốt nhất. Các stage được chạy tuần tự để giải phóng VRAM.

Những stage đã hoàn thành và khớp lineage sẽ được bỏ qua khi chạy lại cell. Stage dở dang yêu cầu RUN_NAME mới; checkpoint không lưu optimizer để resume giữa stage.


In [ ]:
for stage,epochs in EPOCHS.items():
    target=WORK/'adapters'/stage
    if (target/'lineage.json').exists():
        meta=json.loads((target/'lineage.json').read_text())
        assert meta['model']==MODEL and meta['revision']==MODEL_REVISION and meta['epochs']==epochs
        assert meta['dataset_manifest_sha256']==hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest()
        assert meta['world_size']==2 and meta['micro_batch']==meta['eval_batch_per_device']==4
        assert meta['gradient_accumulation']==1
        if stage!='global':
            parent=json.loads((WORK/'adapters/global/lineage.json').read_text())
            assert meta['initial_adapter']['sha256']==parent['final_adapter']['sha256']
        print('Already complete:',stage);continue
    (WORK/'status.json').write_text(json.dumps({'status':'training','stage':stage}))
    args=['--stage',stage,'--data-dir',DATA,'--output-dir',WORK,'--model',MODEL,'--revision',MODEL_REVISION,'--epochs',epochs]
    if stage!='global':args+=['--global-adapter',WORK/'adapters/global']
    train_module('umilife.train_2gpu',args,distributed=True)
    meta=json.loads((target/'lineage.json').read_text())
    assert meta['final_adapter']['finite'] and meta['initial_adapter']['sha256']!=meta['final_adapter']['sha256']
    print(stage,meta['before'],meta['after'],'best step',meta['early_stopping']['best_step'])
(WORK/'status.json').write_text(json.dumps({'status':'training_complete','training':True}))


## Post-training validation: supplied ASR → vLLM batch 4

Train/eval-loss dùng transcript sạch của dữ liệu train. Đánh giá workflow dưới đây dùng ASR cho toàn bộ audio hợp lệ, không fallback sang transcript tham chiếu. Chỉ dùng một GPU cho vLLM; hai GPU được dùng trong các stage fine-tuning.


In [ ]:
args=[ASR_PYTHON,'-u','-m','umilife.transcribe_supplied','--data-root',str(DATA_ROOT),'--work',str(WORK),
      '--model-dir',str(ASR_MODEL_DIR),'--threads',str(ASR_THREADS)]
if VALID_LIMIT is not None:args+=['--limit',str(VALID_LIMIT)]
(WORK/'status.json').write_text(json.dumps({'status':'transcribing_audio'}))
subprocess.run(args,cwd=SOURCE,env=ASR_ENV,check=True)
asr=json.loads((WORK/'asr_report.json').read_text());assert not asr['oracle_fallback']


In [ ]:
args=[INFER_PYTHON,'-u','-m','umilife.infer_2gpu','--data-root',str(DATA_ROOT),'--work',str(WORK),
      '--model',MODEL,'--revision',MODEL_REVISION,'--adapters',str(WORK/'adapters'),'--modes',*MODES]
if VALID_LIMIT is not None:args+=['--limit',str(VALID_LIMIT)]
(WORK/'status.json').write_text(json.dumps({'status':'validating'}))
subprocess.run(args,cwd=SOURCE,env=INFER_ENV,check=True)
report=json.loads((WORK/'vllm_runtime.json').read_text())
assert report['finetuned'] and report['adapter_loaded'] and report['batch_size']==4
assert report['samples_per_mode']==(100 if VALID_LIMIT is None else VALID_LIMIT)
assert report['schema_enforced'] and not report['asr_report']['oracle_fallback']
(WORK/'status.json').write_text(json.dumps({'status':'complete','finetuned':True}))
for mode,metrics in report['metrics'].items():print(mode,json.dumps(metrics,ensure_ascii=False,indent=2))
print('Adapters, lineage, predictions, traces, ASR cache and metrics:',WORK)


## Method and label limitations

Rationale giữ nguyên từ artifact Codex cũ: chủ yếu template giải thích công cụ do Codex viết, kèm một số episode overrides; không phải 3.541 đoạn được viết tay độc lập. Bỏ qua trường partition cũ, chỉ dùng split.txt.

Summarizer dùng nhãn tổng hợp ngắn từ observation cuối, giới hạn 180 token nội dung; đây là câu trả lời theo template, không phải nhãn tóm tắt do người viết. Giữ quy ước cũ zero-action → give_up (bao gồm một số yêu cầu không cần thao tác/hủy), nên taxonomy chưa khớp hoàn toàn policy. Không tự sửa gold valid để che sai lệch này.

Dữ liệu valid đã từng được dùng để phát triển policy; điểm trên tập này là điểm development. Không có ví dụ/nhãn valid nào được đưa vào prompt inference hay optimizer.

Đây là bản thích nghi phương pháp paper với QLoRA, Unsloth và schema-constrained decoding. Kết quả chất lượng chỉ có sau khi bạn chạy notebook. Test CPU không xác nhận được VRAM hoặc tương thích CUDA của bản Qwen 2 GPU này.
